# 02 — Data Cleaning and Preprocessing

> **Objective:** Apply the preprocessing pipeline (imputation, outlier treatment, normalization, encoding), compare before/after, and persist the cleaned dataset.

---

## 1. Setup

Load dependencies and configure project path for importing `src.preprocessing`.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

project_root = Path.cwd().parent
if str(project_root / "src") not in sys.path:
    sys.path.append(str(project_root))

from weather_forecast.preprocessing import run_preprocessing_pipeline
from weather_forecast.parquet_io import write_dataframe_parquet

## 2. Load Raw Dataset

Read the raw weather CSV from `data/raw/`.

In [ ]:
raw_path = project_root / "data" / "raw" / "GlobalWeatherRepository.csv"
df_raw = pd.read_csv(raw_path)

print("Raw path:", raw_path)
print("Raw shape:", df_raw.shape)
display(df_raw.head(3))

## 3. Baseline Profile (Before)

Capture nulls, descriptive stats, and shape before transformations.

In [ ]:
nulls_before = df_raw.isna().sum().sort_values(ascending=False)
null_pct_before = ((nulls_before / len(df_raw)) * 100).round(2)

print("Shape before:", df_raw.shape)
display(nulls_before.head(10))
display(null_pct_before.head(10))
display(df_raw.describe(include="all").transpose().head(10))

## 4. Run Preprocessing Pipeline

Apply imputation, IQR clipping, MinMax normalization, and one-hot encoding.

In [ ]:
df_clean, artifacts = run_preprocessing_pipeline(df_raw)

print("Shape after:", df_clean.shape)
print("Encoded columns count:", len(df_clean.columns))
print("Scaled columns count:", len(artifacts["minmax_columns"]))
print("Excluded from normalization:", artifacts["exclude_from_normalize"])
display(df_clean.head(3))

## 5. Before vs After Comparison

Compare missing values, shape, and outlier diagnostics.

In [ ]:
nulls_after = df_clean.isna().sum().sort_values(ascending=False)

comparison = pd.DataFrame(
    {
        "nulls_before": nulls_before,
        "nulls_after": nulls_after.reindex(nulls_before.index, fill_value=0),
    }
).sort_values("nulls_before", ascending=False)

# Outlier count summary based on IQR bounds (before clipping)
outlier_counts = {}
for col, (low, high) in artifacts["iqr_bounds"].items():
    if col in df_raw.columns:
        outlier_counts[col] = int(((df_raw[col] < low) | (df_raw[col] > high)).sum())

outlier_counts = pd.Series(outlier_counts).sort_values(ascending=False)

print("Shape before -> after:", df_raw.shape, "->", df_clean.shape)
display(comparison.head(15))
display(outlier_counts.head(15))

## 6. Save Cleaned Dataset

Persist the cleaned DataFrame to Parquet format.

In [ ]:
output_path = project_root / "data" / "processed" / "cleaned_weather.parquet"

write_dataframe_parquet(df_clean, output_path, preserve_index=False)

print("Saved:", output_path)
print("File exists:", output_path.exists())

## 7. Summary

**Pipeline decisions:**
- Missing values: median (numeric), mode (categorical)
- Outliers: IQR fences with clipping
- Normalization: MinMax scaling (excludes coordinates and epoch)
- Categorical: one-hot encoding via `pd.get_dummies`

**Next steps:** Consider target/ordinal encoding if cardinality growth becomes problematic.